## Prérequis
1. Créer un virtualenv : `python -m venv venv`
2. Activer le virtualenv : `source venv/bin/activate` pour Mac/Linux ou `venv\Scripts\activate.bat` pour Windows
3. Installer les dépendances : `pip install -r requirements.txt`

## Run
1. Lancer le notebook : `jupyter notebook`
2. Ouvrir le notebook `generate.ipynb`
3. Exécuter le notebook

OU

Directement excécuter dans l'IDE

In [31]:
from fpdf import FPDF
from faker import Faker
import random
import json
import os
from datetime import datetime, timedelta

In [32]:
fake = Faker("fr_FR")

COMPANIES = [
  {"nom": "SNCF", "siret": "35291840800021", "adresse": "12 rue de la Paix, 75001 Paris", "tva": "FR12352918400"},
  {"nom": "Air France", "siret": "35291840035021", "adresse": "8 avenue de Lyon, 69002 Lyon", "tva": "FR48482736190"},
  {"nom": "Orange", "siret": "35291846800021", "adresse": "22 bd Gambetta, 33000 Bordeaux", "tva": "FR61618349200"},
  {"nom": "Auchan", "siret": "35291848960021", "adresse": "22 bd Gambetta, 13000 Marseille", "tva": "FR6161834196"},
]

PRODUITS = [
  ("Prestation de conseil", 200.00),
  ("Développement logiciel", 150.00),
  ("Formation en ligne", 80.00),
  ("Maintenance mensuelle", 120.00),
]

def pick_company(exclude=None):
  """Récupère une entreprie au hasard

  Args:
    exclude (dict): Si passé, permet d'éviter de choisir une entreprise déjà sélectionnée (par exemple une entreprise ne va pas se facturer elle-même)
  """
  pool = [c for c in COMPANIES if c != exclude]
  return random.choice(pool)

In [33]:
vendeur = pick_company()
acheteur = pick_company(exclude=vendeur)

print("Vendeur :", vendeur["nom"])
print("Acheteur :", acheteur["nom"])

Vendeur : Air France
Acheteur : Orange


# Création des dossiers destination

In [34]:
def create_destination_folders():
	# Création dynamique des dossiers
	for split_name in ["train", "test"]:
			for category in ["legitimes", "falsifies"]:
					os.makedirs(f"output/{split_name}/{category}", exist_ok=True)

create_destination_folders()

# Génération de factures

In [35]:
def random_date(start_days_ago=180):
  today = datetime.today()
  delta = random.randint(0, start_days_ago)
  return (today - timedelta(days=delta)).date()

date_emission = random_date()
date_echeance = date_emission + timedelta(days=30)

print("Émission :", date_emission.strftime("%d/%m/%Y"))
print("Échéance :", date_echeance.strftime("%d/%m/%Y"))

Émission : 04/02/2026
Échéance : 06/03/2026


In [36]:
nb_lignes = random.randint(2, 5)
lignes = []

for _ in range(nb_lignes):
    nom, pu = random.choice(PRODUITS)
    qte = random.randint(1, 10)
    total = round(qte * pu, 2)
    lignes.append({"nom": nom, "qte": qte, "pu": pu, "total": total})

print("____________________________________________________________")
for l in lignes:
    print(f"{l['nom']} x{l['qte']} → {l['total']} €")

total_ht = round(sum(l["total"] for l in lignes), 2)
tva = round(total_ht * 0.20, 2)
total_ttc = round(total_ht + tva, 2)
print("____________________________________________________________")
# assert total_ttc == total_ht + tva
print(f"Total HT  : {total_ht} €")
print(f"TVA (20%) : {tva} €")
print(f"Total TTC : {total_ttc} €")
print("____________________________________________________________")

____________________________________________________________
Prestation de conseil x3 → 600.0 €
Maintenance mensuelle x8 → 960.0 €
____________________________________________________________
Total HT  : 1560.0 €
TVA (20%) : 312.0 €
Total TTC : 1872.0 €
____________________________________________________________


In [37]:
def generate_facture(output_path, anomalies=None, vendeur_fixe=None):
    if anomalies is None:
        anomalies = []

    # 1. Données de base
    vendeur  = vendeur_fixe if vendeur_fixe else pick_company()
    acheteur = pick_company(exclude=vendeur)

    if "siret_invalide" in anomalies:
        autre        = pick_company(exclude=vendeur)
        siret_to_show = autre["siret"]
    else:
        siret_to_show = vendeur["siret"]

    # 2. Dates
    date_emission = random_date()
    date_echeance = date_emission + timedelta(days=30)

    # 3. Lignes de produits
    nb_lignes = random.randint(2, 5)
    lignes    = []
    for _ in range(nb_lignes):
        nom, pu = random.choice(PRODUITS)
        qte     = random.randint(1, 10)
        total   = round(qte * pu, 2)
        lignes.append({"nom": nom, "qte": qte, "pu": pu, "total": total})

    # 4. Totaux
    total_ht = round(sum(l["total"] for l in lignes), 2)
    taux_tva = 0.20
    if "tva_incorrecte" in anomalies:
        taux_tva = random.choice([0.05, 0.35, 0.50])
    tva       = round(total_ht * taux_tva, 2)
    total_ttc = round(total_ht + tva, 2)
    if "montant_incoherent" in anomalies:
        total_ttc = round(total_ttc * random.uniform(0.7, 1.4), 2)

    # 5. PDF
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_font("DejaVu",  "",  "fonts/DejaVuSans.ttf")
    pdf.add_font("DejaVu",  "B", "fonts/DejaVuSans-Bold.ttf")

    # Titre
    pdf.set_font("DejaVu", "B", 16)
    pdf.cell(0, 10, "FACTURE", align="C", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)

    # Vendeur
    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 6, "EMETTEUR", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("DejaVu", "", 10)
    pdf.cell(0, 5, vendeur["nom"],                     new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, vendeur["adresse"],                 new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"SIRET : {siret_to_show}",         new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"N TVA : {vendeur['tva']}",        new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)

    # Acheteur
    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 6, "DESTINATAIRE", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("DejaVu", "", 10)
    pdf.cell(0, 5, acheteur["nom"],                    new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, acheteur["adresse"],                new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"SIRET : {acheteur['siret']}",     new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)

    # Infos facture
    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 6, "INFORMATIONS", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("DejaVu", "", 10)
    pdf.cell(0, 5, f"Numero        : FAC-{random.randint(1000, 9999)}",        new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"Date emission : {date_emission.strftime('%d/%m/%Y')}",    new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"Echeance      : {date_echeance.strftime('%d/%m/%Y')}",    new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)

    # En-tête tableau
    pdf.set_font("DejaVu", "B", 10)
    pdf.set_fill_color(220, 220, 220)
    pdf.cell(80, 7, "Designation",   border=1, fill=True)
    pdf.cell(20, 7, "Qte",           border=1, fill=True, align="C")
    pdf.cell(35, 7, "PU HT (EUR)",   border=1, fill=True, align="R")
    pdf.cell(35, 7, "Total HT (EUR)",border=1, fill=True, align="R", new_x="LMARGIN", new_y="NEXT")

    # Lignes produits
    pdf.set_font("DejaVu", "", 10)
    for ligne in lignes:
        pdf.cell(80, 6, ligne["nom"],            border=1)
        pdf.cell(20, 6, str(ligne["qte"]),       border=1, align="C")
        pdf.cell(35, 6, f"{ligne['pu']:.2f}",    border=1, align="R")
        pdf.cell(35, 6, f"{ligne['total']:.2f}", border=1, align="R", new_x="LMARGIN", new_y="NEXT")

    # Totaux
    pdf.ln(4)
    pdf.set_font("DejaVu", "", 10)
    pdf.cell(135, 6, "")
    pdf.cell(35, 6, f"Total HT  : {total_ht:.2f} EUR",              align="R", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(135, 6, "")
    pdf.cell(35, 6, f"TVA ({int(taux_tva*100)}%) : {tva:.2f} EUR",  align="R", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("DejaVu", "B", 10)
    pdf.cell(135, 6, "")
    pdf.cell(35, 6, f"Total TTC : {total_ttc:.2f} EUR",             align="R", new_x="LMARGIN", new_y="NEXT")

    pdf.output(output_path)

    return {
        "fichier":        os.path.basename(output_path),
        "type":           "facture",
        "legitime":       len(anomalies) == 0,
        "format":         "pdf",
        "degradation":    None,
        "anomalies":      anomalies,
        "vendeur":        vendeur["nom"],
        "siret_reel":     vendeur["siret"],
        "siret_affiche":  siret_to_show,
        "acheteur":       acheteur["nom"],
        "total_ht":       total_ht,
        "tva":            tva,
        "taux_tva":       taux_tva,
        "total_ttc":      total_ttc,
        "date":           date_emission.isoformat(),
        "doc_lie":        None,
    }

In [38]:
# Génération vraie facture
meta = generate_facture("output/test/legitimes/facture_test.pdf") 
meta

{'fichier': 'facture_test.pdf',
 'type': 'facture',
 'legitime': True,
 'format': 'pdf',
 'degradation': None,
 'anomalies': [],
 'vendeur': 'Air France',
 'siret_reel': '35291840035021',
 'siret_affiche': '35291840035021',
 'acheteur': 'Orange',
 'total_ht': 1840.0,
 'tva': 368.0,
 'taux_tva': 0.2,
 'total_ttc': 2208.0,
 'date': '2025-11-29',
 'doc_lie': None}

In [39]:
# Génération mauvaise facture
generate_facture("output/test/falsifies/facture_siret.pdf",   anomalies=["siret_invalide"])
generate_facture("output/test/falsifies/facture_tva.pdf",     anomalies=["tva_incorrecte"])
generate_facture("output/test/falsifies/facture_montant.pdf", anomalies=["montant_incoherent"])

{'fichier': 'facture_montant.pdf',
 'type': 'facture',
 'legitime': False,
 'format': 'pdf',
 'degradation': None,
 'anomalies': ['montant_incoherent'],
 'vendeur': 'SNCF',
 'siret_reel': '35291840800021',
 'siret_affiche': '35291840800021',
 'acheteur': 'Auchan',
 'total_ht': 1350.0,
 'tva': 270.0,
 'taux_tva': 0.2,
 'total_ttc': 1969.09,
 'date': '2025-12-30',
 'doc_lie': None}

# Attestation de vigilance

In [40]:
def generate_attestation(output_path, anomalies=None, vendeur_fixe=None):
    if anomalies is None:
        anomalies = []

    # 1. Données de base
    entreprise    = vendeur_fixe if vendeur_fixe else pick_company()
    date_emission = random_date()

    # 2. Date d'expiration
    if "date_expiree" in anomalies:
        date_expiration = date_emission - timedelta(days=random.randint(1, 180))
    else:
        date_expiration = date_emission + timedelta(days=180)

    numero_attestation = f"ATT-URSSAF-{random.randint(100000, 999999)}"

    # 3. PDF
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_font("DejaVu",  "",  "fonts/DejaVuSans.ttf")
    pdf.add_font("DejaVu",  "B", "fonts/DejaVuSans-Bold.ttf")

    # Titre
    pdf.set_font("DejaVu", "B", 16)
    pdf.cell(0, 10, "ATTESTATION DE VIGILANCE", align="C", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("DejaVu", "", 10)
    pdf.cell(0, 6, "Delivree par l'URSSAF", align="C", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(8)

    # Entreprise
    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 6, "ENTREPRISE CONCERNEE", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2)
    pdf.set_font("DejaVu", "", 10)
    pdf.cell(0, 5, f"Raison sociale : {entreprise['nom']}",     new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"SIRET          : {entreprise['siret']}",   new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"Adresse        : {entreprise['adresse']}", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"N TVA          : {entreprise['tva']}",     new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)

    # Validité
    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 6, "PERIODE DE VALIDITE", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2)
    pdf.set_font("DejaVu", "", 10)
    pdf.cell(0, 5, f"Date d'emission    : {date_emission.strftime('%d/%m/%Y')}",   new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"Date d'expiration  : {date_expiration.strftime('%d/%m/%Y')}", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, f"Numero             : {numero_attestation}",                   new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)

    # Mention légale
    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 6, "ATTESTATION", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2)
    pdf.set_font("DejaVu", "", 10)
    pdf.multi_cell(0, 5,
        f"L'URSSAF atteste que l'entreprise {entreprise['nom']} (SIRET : {entreprise['siret']}) "
        f"est a jour de ses obligations de declaration et de paiement des cotisations "
        f"et contributions sociales a la date du {date_emission.strftime('%d/%m/%Y')}. "
        f"La presente attestation est valable jusqu'au {date_expiration.strftime('%d/%m/%Y')}."
    )
    pdf.ln(8)

    # Signature
    pdf.set_font("DejaVu", "B", 10)
    pdf.cell(0, 5, "Pour l'URSSAF", align="R", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("DejaVu", "", 10)
    pdf.cell(0, 5, f"Le {date_emission.strftime('%d/%m/%Y')}", align="R", new_x="LMARGIN", new_y="NEXT")

    pdf.output(output_path)
    print(f"Généré : {output_path}")

    return {
        "fichier":          os.path.basename(output_path),
        "type":             "attestation",
        "legitime":         len(anomalies) == 0,
        "format":           "pdf",
        "degradation":      None,
        "anomalies":        anomalies,
        "entreprise":       entreprise["nom"],
        "siret":            entreprise["siret"],
        "date_emission":    date_emission.isoformat(),
        "date_expiration":  date_expiration.isoformat(),
        "doc_lie":          None,
    }

In [41]:
generate_attestation("output/test/legitimes/attestation_001.pdf")
generate_attestation("output/test/falsifies/attestation_expiree.pdf", anomalies=["date_expiree"])

Généré : output/test/legitimes/attestation_001.pdf
Généré : output/test/falsifies/attestation_expiree.pdf


{'fichier': 'attestation_expiree.pdf',
 'type': 'attestation',
 'legitime': False,
 'format': 'pdf',
 'degradation': None,
 'anomalies': ['date_expiree'],
 'entreprise': 'SNCF',
 'siret': '35291840800021',
 'date_emission': '2026-03-07',
 'date_expiration': '2026-03-05',
 'doc_lie': None}

# Dégradations

In [42]:
import pymupdf
from PIL import Image, ImageFilter

In [43]:
def pdf_to_jpg(pdf_path, output_path, dpi=150):
	doc = pymupdf.open(pdf_path)
	page = doc[0]  # première page uniquement
	mat = pymupdf.Matrix(dpi/72, dpi/72)
	pix = page.get_pixmap(matrix=mat)
	pix.save(output_path)
	doc.close()
	print(f"Converti : {output_path}")

In [44]:
def degrade_blur(img):
	return img.filter(ImageFilter.GaussianBlur(radius=2))

def degrade_rotation(img):
	angle = random.uniform(-5, 5)
	return img.rotate(angle, expand=True, fillcolor="white")

def apply_degradation(jpg_path, output_path, degradation="blur"):
	img = Image.open(jpg_path)

	if degradation == "blur":
		img = degrade_blur(img)

	elif degradation == "rotation":
		img = degrade_rotation(img)

	elif degradation == "both":
		img = degrade_rotation(img)
		img = degrade_blur(img)

	img.save(output_path)
	print(f"Dégradé ({degradation}) : {output_path}")

# Datasets

In [45]:
def generate_dataset(n_legitimes=10, n_falsifies=10, split=0.8):
    create_destination_folders()
    labels = {}

    anomalies_facture = [
        ["siret_invalide"],
        ["tva_incorrecte"],
        ["montant_incoherent"],
        ["siret_invalide", "tva_incorrecte"],
    ]

    # Légitimes — moitié factures, moitié attestations
    n_train_l = int(n_legitimes * split)
    for i in range(0, n_legitimes, 2):
        dossier = "train" if i < n_train_l else "test"
        vendeur = pick_company()

        path_f  = f"output/{dossier}/legitimes/facture_legit_{i+1:03d}.pdf"
        meta_f  = generate_facture(path_f, vendeur_fixe=vendeur)

        path_a  = f"output/{dossier}/legitimes/attestation_legit_{i+2:03d}.pdf"
        meta_a  = generate_attestation(path_a, vendeur_fixe=vendeur)

        meta_f["doc_lie"] = meta_a["fichier"]
        meta_a["doc_lie"] = meta_f["fichier"]

        labels[meta_f["fichier"]] = meta_f
        labels[meta_a["fichier"]] = meta_a

    # Falsifiés — factures avec anomalies + attestations expirées
    n_train_f = int(n_falsifies * split)
    for i in range(n_falsifies):
        dossier = "train" if i < n_train_f else "test"
        if i % 2 == 0:
            path    = f"output/{dossier}/falsifies/facture_falsif_{i+1:03d}.pdf"
            anomalie = anomalies_facture[i % len(anomalies_facture)]
            meta    = generate_facture(path, anomalies=anomalie)
        else:
            path = f"output/{dossier}/falsifies/attestation_falsif_{i+1:03d}.pdf"
            meta = generate_attestation(path, anomalies=["date_expiree"])
        labels[meta["fichier"]] = meta

    # Conversion PDF en JPG + dégradations
    for fichier, meta in list(labels.items()):
        if not fichier.endswith(".pdf"):
            continue

        categorie = "legitimes" if meta["legitime"] else "falsifies"
        pdf_path  = f"output/train/{categorie}/{fichier}"
        if not os.path.exists(pdf_path):
            pdf_path = f"output/test/{categorie}/{fichier}"
        if not os.path.exists(pdf_path):
            continue

        jpg_path = pdf_path.replace(".pdf", ".jpg")
        pdf_to_jpg(pdf_path, jpg_path)

        if random.random() > 0.5:
            degradation = random.choice(["blur", "rotation", "both"])
            deg_path    = jpg_path.replace(".jpg", f"_{degradation}.jpg")
            apply_degradation(jpg_path, deg_path, degradation=degradation)

            deg_meta               = meta.copy()
            deg_meta["fichier"]    = os.path.basename(deg_path)
            deg_meta["format"]     = "jpg"
            deg_meta["degradation"]= degradation
            labels[deg_meta["fichier"]] = deg_meta

    # Sauvegarder labels.json
    with open("output/labels.json", "w", encoding="utf-8") as f:
        json.dump(labels, f, ensure_ascii=False, indent=2)

    print(f"Termine : {len(labels)} entrees dans labels.json")


In [46]:
generate_dataset(n_legitimes=10, n_falsifies=10)

Généré : output/train/legitimes/attestation_legit_002.pdf
Généré : output/train/legitimes/attestation_legit_004.pdf
Généré : output/train/legitimes/attestation_legit_006.pdf
Généré : output/train/legitimes/attestation_legit_008.pdf
Généré : output/test/legitimes/attestation_legit_010.pdf
Généré : output/train/falsifies/attestation_falsif_002.pdf
Généré : output/train/falsifies/attestation_falsif_004.pdf
Généré : output/train/falsifies/attestation_falsif_006.pdf
Généré : output/train/falsifies/attestation_falsif_008.pdf
Généré : output/test/falsifies/attestation_falsif_010.pdf
Converti : output/train/legitimes/facture_legit_001.jpg
Converti : output/train/legitimes/attestation_legit_002.jpg
Dégradé (both) : output/train/legitimes/attestation_legit_002_both.jpg
Converti : output/train/legitimes/facture_legit_003.jpg
Converti : output/train/legitimes/attestation_legit_004.jpg
Dégradé (blur) : output/train/legitimes/attestation_legit_004_blur.jpg
Converti : output/train/legitimes/facture_l